Gerekli importlar

In [13]:
import requests
import re
from youtube_comment_downloader import *
import pandas as pd
import itertools
import time

In [14]:
video_linkleri = [
'https://www.youtube.com/watch?v=MYvZ-v_w0GA'
]

In [15]:
downloader = YoutubeCommentDownloader()
tum_yorum_verileri = []

In [16]:
print(f"{len(video_linkleri)} adet video taranıyor...\n")

1 adet video taranıyor...



In [17]:
# --- VİDEO BAŞLIĞINI ÇEKEN YARDIMCI FONKSİYON ---
def video_basligini_al(url):
    try:
        # Videonun web sayfasına ufak bir istek atıyoruz
        response = requests.get(url)
        # HTML kodları içindeki <title> etiketini buluyoruz
        baslik = re.search(r'<title>(.*?)</title>', response.text).group(1)
        # Başlığın sonundaki " - YouTube" yazısını temizliyoruz
        return baslik.replace(" - YouTube", "").strip()
    except Exception:
        return "Bilinmeyen Video"

In [18]:
# 2. Her bir link için döngü başlatıyoruz
for index, url in enumerate(video_linkleri, 1):
    try:
        print(f"[{index}/{len(video_linkleri)}] Veri çekiliyor: {url}")
        
        # ÖNCE VİDEO BAŞLIĞINI ÇEKİYORUZ
        video_adi = video_basligini_al(url)
        print(f"[{index}/{len(video_linkleri)}] {video_adi}")
        print(f"Yorumlar çekiliyor, lütfen bekleyin...")
        
        # İŞTE EKSİK OLAN VE SENİN SİLDİĞİN SATIR BURASI:
        # Videodan yorumları çek (Popülerliğe göre)
        comments = downloader.get_comments_from_url(url, sort_by=SORT_BY_POPULAR)
        
        # Hafıza değişkenlerimiz (Ana yorumları akılda tutmak için)
        ana_yorum_id = None
        ana_yorum_yazari = None
        
        video_sayaci = 0
        
        # Her videodan kaç yorum alınsın? (Şu an 2000)
        for comment in itertools.islice(comments, 2000):
            is_reply = comment.get('reply', False)
            yorum_id = comment.get('cid', 'Bilinmiyor')
            yazar = comment.get('author', 'Bilinmiyor')
            
            # YANIT KONTROLÜ VE HAFIZA İŞLEMİ
            if not is_reply:
                # Bu bir "Ana Yorum". Bilgilerini hafızaya alıyoruz.
                ana_yorum_id = yorum_id
                ana_yorum_yazari = yazar
                yanit_edilen_kisi = None
                ust_yorum_id = None
            else:
                # Bu bir "Yanıt". Hafızadaki son ana yoruma aittir.
                yanit_edilen_kisi = ana_yorum_yazari
                ust_yorum_id = ana_yorum_id
            
            # Verileri listeye ekle
            tum_yorum_verileri.append({
                'Video Başlığı': video_adi,
                'Video Kaynağı': url,
                'Yorum ID': yorum_id,
                'Üst Yorum ID': ust_yorum_id,
                'Yorum Yapan Kişi': yazar,
                'Yanıtlanan Kişi': yanit_edilen_kisi,
                'Yorum Metni': comment.get('text', ''),
                'Beğeni Sayısı': comment.get('votes', 0),
                'Tarih': comment.get('time', 'Bilinmiyor'),
                'Yanıt Mı?': 'Evet' if is_reply else 'Hayır'
            })
            video_sayaci += 1
        
        print(f"Başarılı! Bu videodan {video_sayaci} yorum alındı.\n")
        
        # YouTube bot korumasına takılmamak için kısa bir bekleme
        time.sleep(1) 
        
    except Exception as e:
        print(f"Hata oluştu ({url}): {e}\n")

[1/1] Veri çekiliyor: https://www.youtube.com/watch?v=MYvZ-v_w0GA
[1/1] Kamera Kayıt Aşk (214. Bölüm) - Çok Güzel Hareketler 2
Yorumlar çekiliyor, lütfen bekleyin...
Başarılı! Bu videodan 181 yorum alındı.



In [19]:
# 3. Tüm verileri Pandas ile tabloya çevirip kaydediyoruz
df = pd.DataFrame(tum_yorum_verileri)

In [20]:
# Türkçe karakter desteğiyle kaydet (Excel'de bozulmaması için utf-8-sig)
df.to_csv('test.csv', index=False, encoding='utf-8-sig')

In [21]:
print("="*40)
print(f"İşlem Tamamlandı!")
print(f"Toplam çekilen yorum sayısı: {len(df)}")
print(f"Dosya adı: test.csv")
print("="*40)

İşlem Tamamlandı!
Toplam çekilen yorum sayısı: 181
Dosya adı: test.csv


In [22]:
df

,Video Başlığı,Video Kaynağı,Yorum ID,Üst Yorum ID,Yorum Yapan Kişi,Yanıtlanan Kişi,Yorum Metni,Beğeni Sayısı,Tarih,Yanıt Mı?
0,Kamera Kayıt Aşk (214. Bölüm) - Çok Güzel Hare...,https://www.youtube.com/watch?v=MYvZ-v_w0GA,UgwDdAfS10YaA-3G5DN4AaABAg,None,@dils4d,None,Sonu çok ters köşeydi olum,846,2 ay önce,Hayır
1,Kamera Kayıt Aşk (214. Bölüm) - Çok Güzel Hare...,https://www.youtube.com/watch?v=MYvZ-v_w0GA,UgwtklRJ7AhTAhfntP94AaABAg,None,@HeyWeAreTH,None,Evliya ve penguenin kendi seçimiydi peki ya wi...,700,2 ay önce,Hayır
2,Kamera Kayıt Aşk (214. Bölüm) - Çok Güzel Hare...,https://www.youtube.com/watch?v=MYvZ-v_w0GA,UgyXSbbrVFL6yQYicLN4AaABAg,None,@carullahtarim166,None,Ebrunun kalemi çok güçlü yazdığı skeçleri izle...,377,2 ay önce,Hayır
3,Kamera Kayıt Aşk (214. Bölüm) - Çok Güzel Hare...,https://www.youtube.com/watch?v=MYvZ-v_w0GA,UgxKLTL7puGSf5cbqrx4AaABAg,None,@n.a.2084,None,Evliyanın teklif metni çok güzeldi,259,2 ay önce,Hayır
4,Kamera Kayıt Aşk (214. Bölüm) - Çok Güzel Hare...,https://www.youtube.com/watch?v=MYvZ-v_w0GA,UgxA3o3gwjmYkg0wME54AaABAg,None,@Salllllimjk,None,Beyinsiz seyircinin sürekli alkislamasi çok it...,122,2 ay önce,Hayır
...,...,...,...,...,...,...,...,...,...,...
176,Kamera Kayıt Aşk (214. Bölüm) - Çok Güzel Hare...,https://www.youtube.com/watch?v=MYvZ-v_w0GA,Ugy7ntc6Oh5s0tXTo7d4AaABAg.ATfKF-t8G1QATu8tOxr0Mn,UgwD06Wmp9Zqb1A5PAp4AaABAg,@CennetSali,@Erenaslan321,Anneninki gibi,0,2 ay önce,Evet
177,Kamera Kayıt Aşk (214. Bölüm) - Çok Güzel Hare...,https://www.youtube.com/watch?v=MYvZ-v_w0GA,Ugw3Ia62opc9kY8d0FV4AaABAg.ATZtja0h9zPAT_AmHdOgYJ,UgwD06Wmp9Zqb1A5PAp4AaABAg,@hayalndünyası4535,@Erenaslan321,Sapikmisin,15,2 ay önce,Evet
178,Kamera Kayıt Aşk (214. Bölüm) - Çok Güzel Hare...,https://www.youtube.com/watch?v=MYvZ-v_w0GA,Ugx4HlufSkspI6m7FN14AaABAg.ATZWHdoDcM8ATZp30x4DZM,UgwD06Wmp9Zqb1A5PAp4AaABAg,@Soldieraga,@Erenaslan321,Okuluna gitsene,13,2 ay önce,Evet
179,Kamera Kayıt Aşk (214. Bölüm) - Çok Güzel Hare...,https://www.youtube.com/watch?v=MYvZ-v_w0GA,Ugx4HlufSkspI6m7FN14AaABAg.ATZWHdoDcM8ATcO904Clps,UgwD06Wmp9Zqb1A5PAp4AaABAg,@hmzawa,@Erenaslan321,Youtube kids,3,2 ay önce,Evet


In [23]:
# İlk satırdaki (0. index) 'Yorum Metni' sütununun tamamını yazdırır
print(df['Yorum Metni'].iloc[0])

Sonu çok ters köşeydi olum
